In [ ]:
# Cell 1: Install dependencies
!pip install -q diffusers transformers accelerate safetensors flask
!pip install -q huggingface_hub omegaconf
!npm install -g localtunnel > /dev/null 2>&1
print('✅ Dependencies installed')

In [ ]:
# Cell 2: Load models (multi-model support)
import torch, gc
from diffusers import StableDiffusionXLPipeline, StableDiffusionPipeline, EulerAncestralDiscreteScheduler

MODELS = {
    'pony-v6': {
        'id': 'John6666/pony-diffusion-v6-xl-sdxl',
        'type': 'xl',
        'label': 'Pony Diffusion V6 XL'
    },
    'illustrious': {
        'id': 'OnomaAIResearch/Illustrious-xl-early-release-v0',
        'type': 'xl',
        'label': 'Illustrious XL'
    },
    'aom3': {
        'id': 'John6666/aom3a1b-sdxl',
        'type': 'xl',
        'label': 'AOM3 XL'
    },
    'meinamix': {
        'id': 'Meina/MeinaMix_V11',
        'type': 'sd15',
        'label': 'MeinaMix V11'
    }
}

pipes = {}
current_model = None
current_pipe = None

def load_model(model_key):
    global current_model, current_pipe
    if model_key == current_model and current_pipe is not None:
        return current_pipe
    
    # Free previous model
    if current_pipe is not None:
        del current_pipe
        gc.collect()
        torch.cuda.empty_cache()
    
    info = MODELS[model_key]
    print(f'Loading {info["label"]}...')
    
    if info['type'] == 'xl':
        pipe = StableDiffusionXLPipeline.from_pretrained(
            info['id'],
            torch_dtype=torch.float16,
            use_safetensors=True,
            variant='fp16'
        )
    else:
        pipe = StableDiffusionPipeline.from_pretrained(
            info['id'],
            torch_dtype=torch.float16,
            safety_checker=None,
            requires_safety_checker=False
        )
    
    pipe.scheduler = EulerAncestralDiscreteScheduler.from_config(pipe.scheduler.config)
    pipe = pipe.to('cuda')
    pipe.enable_attention_slicing()
    
    current_model = model_key
    current_pipe = pipe
    print(f'✅ {info["label"]} loaded on GPU!')
    return pipe

# Load default model
load_model('pony-v6')
print('\n🎨 Multi-model system ready!')

In [ ]:
# Cell 3: Start API server + tunnel
from flask import Flask, request, send_file, jsonify, Response
import io, subprocess, threading, time, re

app = Flask(__name__)

@app.after_request
def add_cors(response):
    response.headers['Access-Control-Allow-Origin'] = '*'
    response.headers['Access-Control-Allow-Headers'] = '*'
    response.headers['Access-Control-Allow-Methods'] = '*'
    return response

@app.route('/models', methods=['GET'])
def list_models():
    models = []
    for key, info in MODELS.items():
        models.append({
            'id': key,
            'label': info['label'],
            'type': info['type'],
            'active': key == current_model
        })
    return jsonify({'models': models, 'current': current_model})

@app.route('/switch', methods=['POST', 'OPTIONS'])
def switch_model():
    if request.method == 'OPTIONS':
        return Response(status=200)
    data = request.json
    model_key = data.get('model', 'pony-v6')
    if model_key not in MODELS:
        return jsonify({'error': f'Unknown model: {model_key}'}), 400
    load_model(model_key)
    return jsonify({'status': 'ok', 'model': model_key, 'label': MODELS[model_key]['label']})

@app.route('/generate', methods=['POST', 'OPTIONS'])
def generate():
    if request.method == 'OPTIONS':
        return Response(status=200)
    data = request.json
    prompt = data.get('prompt', '')
    negative = data.get('negative_prompt', 'ugly, deformed, blurry, low quality, bad anatomy, censored, clothed, text, watermark')
    steps = min(data.get('steps', 25), 50)
    width = data.get('width', 512)
    height = data.get('height', 512)
    model_key = data.get('model', current_model)
    
    # Switch model if needed
    if model_key != current_model:
        load_model(model_key)
    
    full_prompt = f'{prompt}, masterpiece, best quality, detailed, anime'
    
    pipe = current_pipe
    image = pipe(
        full_prompt,
        negative_prompt=negative,
        num_inference_steps=steps,
        guidance_scale=7.5,
        width=width,
        height=height
    ).images[0]
    
    buf = io.BytesIO()
    image.save(buf, format='PNG')
    buf.seek(0)
    return send_file(buf, mimetype='image/png')

@app.route('/health')
def health():
    return jsonify({'status': 'ok', 'model': current_model, 'label': MODELS.get(current_model, {}).get('label', 'unknown')})

# Start Flask in background
def run_flask():
    app.run(port=5000, host='0.0.0.0')

t = threading.Thread(target=run_flask, daemon=True)
t.start()
time.sleep(2)

# Start localtunnel
proc = subprocess.Popen(['lt', '--port', '5000'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
time.sleep(5)
line = proc.stdout.readline()
print(f'\n🔥 ProjetH API is live!')
print(f'URL: {line.strip()}')
print(f'Current model: {MODELS[current_model]["label"]}')
print(f'\nEndpoints:')
print(f'  GET  /health  - Status check')
print(f'  GET  /models  - List available models')
print(f'  POST /switch  - Switch model {{"model": "pony-v6"}}')
print(f'  POST /generate - Generate image')
print(f'\nCopy the URL into the web app!')

# Keep alive
while True:
    time.sleep(60)